# Silver Tables to Gold Star Schema

この Notebook は Lakehouse 上の `silver.City`、`silver.Sale`、`silver.Stock_Item` を直接読み込み、Gold スキーマに `FactSales` / `DimCity` / `DimStockItem` を作成します。

## この Notebook の前提
- 入力テーブル: `silver.City`, `silver.Sale`, `silver.Stock_Item`
- 出力テーブル: `gold.FactSales`, `gold.DimCity`, `gold.DimStockItem`
- `silver.Sale` を Fact の粒度とし、City / Stock Item をディメンション化します
- `DimDate` は別 Notebook または別処理で作成済みであることを前提とします
- `Customer`, `Bill To Customer`, `Salesperson` のディメンションは元データ未提供のため、この Notebook では作成しません

## この Notebook で実施すること
1. Silver テーブルを読み込み、型を補正する
2. Fact が参照するキーだけに絞って `DimCity` と `DimStockItem` を作る
3. `Sale` を `FactSales` に整形し、`InvoiceDateKey` / `DeliveryDateKey` を付与する
4. Gold スキーマに保存し、件数と参照整合性を確認する

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, BooleanType
from pyspark.sql.window import Window

LAKEHOUSE_NAME = "demo_lakehouse"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
DIM_CITY_TABLE = "DimCity"
DIM_STOCK_ITEM_TABLE = "DimStockItem"
FACT_SALES_TABLE = "FactSales"

SOURCE_CITY_TABLE = "City"
SOURCE_SALE_TABLE = "Sale"
SOURCE_STOCK_ITEM_TABLE = "Stock_Item"

def build_table_fqn(schema_name, table_name):
    if LAKEHOUSE_NAME:
        return f"{LAKEHOUSE_NAME}.{schema_name}.{table_name}"
    return f"{schema_name}.{table_name}"

CITY_SOURCE_FQN = build_table_fqn(SILVER_SCHEMA, SOURCE_CITY_TABLE)
SALE_SOURCE_FQN = build_table_fqn(SILVER_SCHEMA, SOURCE_SALE_TABLE)
STOCK_ITEM_SOURCE_FQN = build_table_fqn(SILVER_SCHEMA, SOURCE_STOCK_ITEM_TABLE)

DIM_CITY_PATH = f"Tables/{GOLD_SCHEMA}/{DIM_CITY_TABLE}"
DIM_STOCK_ITEM_PATH = f"Tables/{GOLD_SCHEMA}/{DIM_STOCK_ITEM_TABLE}"
FACT_SALES_PATH = f"Tables/{GOLD_SCHEMA}/{FACT_SALES_TABLE}"

print("LAKEHOUSE_NAME:", LAKEHOUSE_NAME)
print("SILVER_SCHEMA:", SILVER_SCHEMA)
print("GOLD_SCHEMA:", GOLD_SCHEMA)
print("CITY_SOURCE_FQN:", CITY_SOURCE_FQN)
print("SALE_SOURCE_FQN:", SALE_SOURCE_FQN)
print("STOCK_ITEM_SOURCE_FQN:", STOCK_ITEM_SOURCE_FQN)

## Silver テーブルの読み込みと型補正
Lakehouse 上の `silver` テーブルを直接読み込み、Power BI 向け Gold モデルで扱いやすいように基本的な型を整えます。

In [ ]:
def read_silver_table(table_fqn):
    return spark.sql(f"SELECT * FROM {table_fqn}")

def transform_city(df):
    return (
        df
        .withColumn("City_Key", F.col("City_Key").cast(IntegerType()))
        .withColumn("WWI_City_ID", F.col("WWI_City_ID").cast(IntegerType()))
        .withColumn("Latest_Recorded_Population", F.col("Latest_Recorded_Population").cast(IntegerType()))
        .withColumn("Valid_From", F.to_timestamp("Valid_From"))
        .withColumn("Valid_To", F.to_timestamp("Valid_To"))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
        .withColumn("ProcessedTimestamp", F.to_timestamp("ProcessedTimestamp"))
    )

def transform_sale(df):
    return (
        df
        .withColumn("Sale_Key", F.col("Sale_Key").cast(IntegerType()))
        .withColumn("City_Key", F.col("City_Key").cast(IntegerType()))
        .withColumn("Customer_Key", F.col("Customer_Key").cast(IntegerType()))
        .withColumn("Bill_To_Customer_Key", F.col("Bill_To_Customer_Key").cast(IntegerType()))
        .withColumn("Stock_Item_Key", F.col("Stock_Item_Key").cast(IntegerType()))
        .withColumn("Salesperson_Key", F.col("Salesperson_Key").cast(IntegerType()))
        .withColumn("WWI_Invoice_ID", F.col("WWI_Invoice_ID").cast(IntegerType()))
        .withColumn("Invoice_Date_Key", F.to_date("Invoice_Date_Key"))
        .withColumn("Delivery_Date_Key", F.to_date("Delivery_Date_Key"))
        .withColumn("Quantity", F.col("Quantity").cast(IntegerType()))
        .withColumn("Unit_Price", F.col("Unit_Price").cast(DoubleType()))
        .withColumn("Tax_Rate", F.col("Tax_Rate").cast(DoubleType()))
        .withColumn("Total_Excluding_Tax", F.col("Total_Excluding_Tax").cast(DoubleType()))
        .withColumn("Tax_Amount", F.col("Tax_Amount").cast(DoubleType()))
        .withColumn("Profit", F.col("Profit").cast(DoubleType()))
        .withColumn("Total_Including_Tax", F.col("Total_Including_Tax").cast(DoubleType()))
        .withColumn("Total_Dry_Items", F.col("Total_Dry_Items").cast(DoubleType()))
        .withColumn("Total_Chiller_Items", F.col("Total_Chiller_Items").cast(DoubleType()))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
        .withColumn("ProcessedTimestamp", F.to_timestamp("ProcessedTimestamp"))
    )

def transform_stock_item(df):
    return (
        df
        .withColumn("Stock_Item_Key", F.col("Stock_Item_Key").cast(IntegerType()))
        .withColumn("WWI_Stock_Item_ID", F.col("WWI_Stock_Item_ID").cast(IntegerType()))
        .withColumn("Lead_Time_Days", F.col("Lead_Time_Days").cast(IntegerType()))
        .withColumn("Quantity_Per_Outer", F.col("Quantity_Per_Outer").cast(IntegerType()))
        .withColumn("Is_Chiller_Stock", F.col("Is_Chiller_Stock").cast(BooleanType()))
        .withColumn("Tax_Rate", F.col("Tax_Rate").cast(DoubleType()))
        .withColumn("Unit_Price", F.col("Unit_Price").cast(DoubleType()))
        .withColumn("Recommended_Retail_Price", F.col("Recommended_Retail_Price").cast(DoubleType()))
        .withColumn("Typical_Weight_Per_Unit", F.col("Typical_Weight_Per_Unit").cast(DoubleType()))
        .withColumn("Valid_From", F.to_timestamp("Valid_From"))
        .withColumn("Valid_To", F.to_timestamp("Valid_To"))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
    )

city_silver = transform_city(read_silver_table(CITY_SOURCE_FQN))
sale_silver = transform_sale(read_silver_table(SALE_SOURCE_FQN))
stock_item_silver = transform_stock_item(read_silver_table(STOCK_ITEM_SOURCE_FQN))

display(city_silver.limit(10))
display(sale_silver.limit(10))
display(stock_item_silver.limit(10))

## 入力データの概要確認
Fact の件数、日付範囲、空の Delivery Date 件数を確認します。

In [ ]:
input_profile = [
    ("City", city_silver.count(), city_silver.select('City_Key').distinct().count()),
    ("Sale", sale_silver.count(), sale_silver.select('Sale_Key').distinct().count()),
    ("Stock_Item", stock_item_silver.count(), stock_item_silver.select('Stock_Item_Key').distinct().count())
]
display(spark.createDataFrame(input_profile, ['TableName', 'RowCount', 'DistinctPrimaryKeyCount']))

sale_date_summary = sale_silver.agg(
    F.min('Invoice_Date_Key').alias('MinInvoiceDate'),
    F.max('Invoice_Date_Key').alias('MaxInvoiceDate'),
    F.min('Delivery_Date_Key').alias('MinDeliveryDate'),
    F.max('Delivery_Date_Key').alias('MaxDeliveryDate'),
    F.sum(F.when(F.col('Delivery_Date_Key').isNull(), 1).otherwise(0)).alias('NullDeliveryDateCount')
)
display(sale_date_summary)

## Gold ディメンションの作成
`DimCity` と `DimStockItem` は、Fact が参照するキーだけに絞って作成します。

履歴列 `Valid_From` / `Valid_To` は SCD 管理用途のため公開モデルからは外し、分析に必要な属性だけ残します。

In [ ]:
referenced_city_keys = sale_silver.select('City_Key').distinct()
referenced_stock_item_keys = sale_silver.select('Stock_Item_Key').distinct()

city_window = Window.partitionBy('City_Key').orderBy(F.col('Valid_To').desc_nulls_last(), F.col('ProcessedTimestamp').desc_nulls_last())
stock_item_window = Window.partitionBy('Stock_Item_Key').orderBy(F.col('Valid_To').desc_nulls_last(), F.col('Valid_From').desc_nulls_last())

dim_city = (
    city_silver
    .join(referenced_city_keys, on='City_Key', how='inner')
    .withColumn('RowPriority', F.row_number().over(city_window))
    .filter(F.col('RowPriority') == 1)
    .select(
        F.col('City_Key').alias('CityKey'),
        F.col('WWI_City_ID').alias('WWICityId'),
        F.col('City').alias('CityName'),
        F.col('State_Province').alias('StateProvince'),
        F.col('Country'),
        F.col('Continent'),
        F.col('Sales_Territory').alias('SalesTerritory'),
        F.col('Region'),
        F.col('Subregion'),
        F.col('Latest_Recorded_Population').alias('LatestRecordedPopulation')
    )
)

dim_stock_item = (
    stock_item_silver
    .join(referenced_stock_item_keys, on='Stock_Item_Key', how='inner')
    .withColumn('RowPriority', F.row_number().over(stock_item_window))
    .filter(F.col('RowPriority') == 1)
    .select(
        F.col('Stock_Item_Key').alias('StockItemKey'),
        F.col('WWI_Stock_Item_ID').alias('WWIStockItemId'),
        F.col('Stock_Item').alias('StockItemName'),
        F.col('Color'),
        F.col('Selling_Package').alias('SellingPackage'),
        F.col('Buying_Package').alias('BuyingPackage'),
        F.col('Brand'),
        F.col('Size'),
        F.col('Lead_Time_Days').alias('LeadTimeDays'),
        F.col('Quantity_Per_Outer').alias('QuantityPerOuter'),
        F.col('Is_Chiller_Stock').alias('IsChillerStock'),
        F.col('Tax_Rate').alias('TaxRate'),
        F.col('Unit_Price').alias('UnitPrice'),
        F.col('Recommended_Retail_Price').alias('RecommendedRetailPrice'),
        F.col('Typical_Weight_Per_Unit').alias('TypicalWeightPerUnit'),
        F.col('Product_Category').alias('ProductCategory')
    )
)

display(dim_city.limit(20))
display(dim_stock_item.limit(20))

## 日付キーの確認
この Notebook では `DimDate` 自体は作成せず、`FactSales` に格納する `InvoiceDateKey` / `DeliveryDateKey` の元になる日付範囲だけ確認します。

In [ ]:
fact_date_keys_profile = sale_silver.agg(
    F.min('Invoice_Date_Key').alias('MinInvoiceDate'),
    F.max('Invoice_Date_Key').alias('MaxInvoiceDate'),
    F.min('Delivery_Date_Key').alias('MinDeliveryDate'),
    F.max('Delivery_Date_Key').alias('MaxDeliveryDate'),
    F.countDistinct('Invoice_Date_Key').alias('DistinctInvoiceDateCount'),
    F.countDistinct('Delivery_Date_Key').alias('DistinctDeliveryDateCount'),
    F.sum(F.when(F.col('Delivery_Date_Key').isNull(), 1).otherwise(0)).alias('NullDeliveryDateCount')
)

display(fact_date_keys_profile)

fact_date_key_samples = (
    sale_silver
    .select(
        F.date_format('Invoice_Date_Key', 'yyyyMMdd').cast(IntegerType()).alias('InvoiceDateKey'),
        F.when(
            F.col('Delivery_Date_Key').isNotNull(),
            F.date_format('Delivery_Date_Key', 'yyyyMMdd').cast(IntegerType())
        ).otherwise(F.lit(None).cast(IntegerType())).alias('DeliveryDateKey')
    )
    .distinct()
    .orderBy('InvoiceDateKey', 'DeliveryDateKey')
)

display(fact_date_key_samples.limit(20))

## FactSales の作成
監査列や Silver 管理列は落とし、分析向けのキー列と指標列に整理します。

`CustomerKey` / `BillToCustomerKey` / `SalespersonKey` は将来ディメンションを追加できるよう Fact に残します。

In [ ]:
fact_sales = (
    sale_silver
    .select(
        F.col('Sale_Key').alias('SaleKey'),
        F.col('City_Key').alias('CityKey'),
        F.col('Stock_Item_Key').alias('StockItemKey'),
        F.date_format('Invoice_Date_Key', 'yyyyMMdd').cast(IntegerType()).alias('InvoiceDateKey'),
        F.when(F.col('Delivery_Date_Key').isNotNull(), F.date_format('Delivery_Date_Key', 'yyyyMMdd').cast(IntegerType())).otherwise(F.lit(None).cast(IntegerType())).alias('DeliveryDateKey'),
        F.col('Customer_Key').alias('CustomerKey'),
        F.col('Bill_To_Customer_Key').alias('BillToCustomerKey'),
        F.col('Salesperson_Key').alias('SalespersonKey'),
        F.col('WWI_Invoice_ID').alias('WWIInvoiceId'),
        F.col('Quantity'),
        F.col('Unit_Price').alias('UnitPrice'),
        F.col('Tax_Rate').alias('TaxRate'),
        F.col('Total_Excluding_Tax').alias('TotalExcludingTax'),
        F.col('Tax_Amount').alias('TaxAmount'),
        F.col('Profit'),
        F.col('Total_Including_Tax').alias('TotalIncludingTax'),
        F.col('Total_Dry_Items').alias('TotalDryItems'),
        F.col('Total_Chiller_Items').alias('TotalChillerItems')
    )
)

display(fact_sales.limit(20))

## Gold スキーマへの保存
作成した `DimCity`、`DimStockItem`、`FactSales` を Gold スキーマに保存します。

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

dim_city.write.mode('overwrite').save(DIM_CITY_PATH)
dim_stock_item.write.mode('overwrite').save(DIM_STOCK_ITEM_PATH)
fact_sales.write.mode('overwrite').save(FACT_SALES_PATH)

save_results = [
    (DIM_CITY_TABLE, DIM_CITY_PATH, dim_city.count()),
    (DIM_STOCK_ITEM_TABLE, DIM_STOCK_ITEM_PATH, dim_stock_item.count()),
    (FACT_SALES_TABLE, FACT_SALES_PATH, fact_sales.count())
]
display(spark.createDataFrame(save_results, ['TableName', 'Path', 'RowCount']))

## 検証
Fact と Gold ディメンションの参照整合性を確認します。`DimDate` との整合性確認は別 Notebook 側で実施します。

In [ ]:
missing_city_refs = fact_sales.join(dim_city.select('CityKey'), on='CityKey', how='left_anti').count()
missing_stock_item_refs = fact_sales.join(dim_stock_item.select('StockItemKey'), on='StockItemKey', how='left_anti').count()
null_invoice_date_keys = fact_sales.filter(F.col('InvoiceDateKey').isNull()).count()
null_delivery_date_keys = fact_sales.filter(F.col('DeliveryDateKey').isNull()).count()

validation_results = [
    ('FactSales', fact_sales.count()),
    ('DimCity', dim_city.count()),
    ('DimStockItem', dim_stock_item.count()),
    ('Missing City references', missing_city_refs),
    ('Missing StockItem references', missing_stock_item_refs),
    ('Null InvoiceDateKey', null_invoice_date_keys),
    ('Null DeliveryDateKey', null_delivery_date_keys)
]
display(spark.createDataFrame(validation_results, ['CheckName', 'Value']))

## 補足
- `City` と `Stock_Item` は Fact が参照するキーだけを Gold に持ち込みます。
- `Valid_From`, `Valid_To`, `Lineage_Key`, `ProcessedTimestamp`, `DataQualityFlag` は分析公開モデルから除外しています。
- `Customer`, `Bill To Customer`, `Salesperson` の Gold ディメンションを追加したい場合は、対応する Silver ソースが別途必要です。
- `DimDate` は別 Notebook または別処理で作成し、この Notebook で生成した `InvoiceDateKey` / `DeliveryDateKey` と関連付けてください。